Heterogeneity Heatmaps — 4 statistics x 3 sigma values  
Layout : one 4x3 figure without baseline, one with baseline  
Y-axis : Number of generations after intervention = 36 - t2  
Title  : Constant/Seasonal 99%/90% - With/Without Baseline  
Runs for: Constant_0.01 / Constant_0.1 / Seasonal_0.1

In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)
library(patchwork)

# -- Global settings -----------------------------------------------------------
set.seed(2025)
alpha  <- 0.05
n_sim  <- 1000
k_vec  <- 2:20

# Statistics: display label + plotmath expression (avoids pi/sigma dot bug)
stat_info <- list(
  list(label      = "LD",
       yaxis_expr = expression(atop(bold("LD"),
                              "Generations after crash" ~ (36-t[2]))),
       file = "LD.csv"),

  list(label      = "Tajima's D",
       yaxis_expr = expression(atop(bold("Tajima's D"),
                                    "Generations after crash" ~ (36-t[2]))),
       file = "Tajimas_D.csv"),

  list(label      = "Segregating sites",
       yaxis_expr = expression(atop(bold("Segregating sites"),
                                    "Generations after crash" ~ (36-t[2]))),
       file = "Density_of_segregating_sites.csv"),

  list(label      = "Nucleotide diversity (pi)",
       yaxis_expr = expression(atop(bold(paste("Nucleotide diversity ", pi)),
                                    "Generations after crash" ~ (36-t[2]))),
       file = "Nucleotide_diversity_pi.csv")
)

# Sigma levels (sigma=0.3 removed).
# Column labels use plotmath so sigma renders correctly on all devices.
sigma_dirs        <- c("hetero_0", "hetero_0.1", "hetero_0.5")
sigma_label_exprs <- list(
  expression(paste(sigma, " = 0")),
  expression(paste(sigma, " = 0.1")),
  expression(paste(sigma, " = 0.5"))
)

# Map folder name
dataset_label_map <- list(
  "Constant_0.01" = "Constant 99% crash",
  "Constant_0.1"  = "Constant 90% crash",
  "Seasonal_0.1"  = "Seasonal 90% crash"
)

# Dataset roots
dataset_roots <- c(
  "../data/Constant_0.01",
  "../data/Constant_0.1",
  "../data/Seasonal_0.1"
)

# Font / legend sizes (2x larger than previous version)
BASE_TITLE   <- 44   # column header (sigma label)
BASE_AXIS_T  <- 28   # axis titles
BASE_AXIS_TX <- 16   # axis tick text
BASE_LEG_T   <- 30   # legend title
BASE_LEG_TX  <- 22   # legend text
BASE_ANNOT   <- 28   # figure-level annotation title
BASE_KEY     <- 2  # legend key height in cm

Power estimation

In [ ]:
# -- Power-estimation helpers --------------------------------------------------

estimate_power_no_baseline <- function(data, t2, k,
                                       n_sim = 1000, alpha = 0.05,
                                       model_control    = "control",
                                       model_interv     = "intervention",
                                       replace_sampling = FALSE,
                                       use_var_equal    = FALSE) {
  sub      <- data %>% filter(Time == t2)
  ctrl_v   <- sub %>% filter(Model == model_control) %>% pull(Value)
  interv_v <- sub %>% filter(Model == model_interv)  %>% pull(Value)

  reject <- logical(n_sim)
  for (i in seq_len(n_sim)) {
    sc <- sample(ctrl_v,   size = k, replace = replace_sampling)
    si <- sample(interv_v, size = k, replace = replace_sampling)
    tt <- try(t.test(si, sc, var.equal = use_var_equal), silent = TRUE)
    reject[i] <- if (inherits(tt, "try-error") || is.null(tt$p.value)) FALSE
                 else tt$p.value < alpha
  }
  mean(reject, na.rm = TRUE)
}

estimate_power_baseline_diff <- function(data, t1 = 39, t2, k,
                                         n_sim = 1000, alpha = 0.05,
                                         model_control    = "control",
                                         model_interv     = "intervention",
                                         replace_sampling = FALSE,
                                         use_var_equal    = FALSE) {
  d1 <- data %>% filter(Time == t1) %>% arrange(Model, Seed)
  d2 <- data %>% filter(Time == t2) %>% arrange(Model, Seed)

  pool     <- tibble(Model = d1$Model, Diff = d2$Value - d1$Value)
  ctrl_d   <- pool %>% filter(Model == model_control) %>% pull(Diff)
  interv_d <- pool %>% filter(Model == model_interv)  %>% pull(Diff)

  reject <- logical(n_sim)
  for (i in seq_len(n_sim)) {
    sc <- sample(ctrl_d,   size = k, replace = replace_sampling)
    si <- sample(interv_d, size = k, replace = replace_sampling)
    tt <- try(t.test(si, sc, var.equal = use_var_equal), silent = TRUE)
    reject[i] <- if (inherits(tt, "try-error") || is.null(tt$p.value)) FALSE
                 else tt$p.value < alpha
  }
  mean(reject, na.rm = TRUE)
}

# -- Compute power grid for one data frame -------------------------------------

compute_power_grid <- function(df, with_baseline) {
  t2_values <- df %>%
    distinct(Time) %>% pull(Time) %>% sort()
  t2_values <- t2_values[t2_values < 36]

  grid <- expand.grid(time = t2_values, k = k_vec, stringsAsFactors = FALSE) %>%
    arrange(time, k) %>%
    mutate(power = NA_real_)

  for (r in seq_len(nrow(grid))) {
    grid$power[r] <- if (with_baseline) {
      estimate_power_baseline_diff(data = df, t1 = 39,
                                   t2 = grid$time[r], k = grid$k[r],
                                   n_sim = n_sim, alpha = alpha)
    } else {
      estimate_power_no_baseline(data = df,
                                 t2 = grid$time[r], k = grid$k[r],
                                 n_sim = n_sim, alpha = alpha)
    }
  }

  grid <- grid %>% mutate(gen_after = 36 - time)
  grid
}


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




Processing: Constant 99% crash  (Constant_0.01)


  Sigma dir: hetero_0

    Reading: LD.csv

    Computing power (no baseline)...

    Computing power (with baseline)...

    Reading: Tajimas_D.csv

    Computing power (no baseline)...

    Computing power (with baseline)...

    Reading: Density_of_segregating_sites.csv

    Computing power (no baseline)...

    Computing power (with baseline)...

    Reading: Nucleotide_diversity_pi.csv

    Computing power (no baseline)...

    Computing power (with baseline)...

  Sigma dir: hetero_0.1

    Reading: LD.csv

    Computing power (no baseline)...

    Computing power (with baseline)...

    Reading: Tajimas_D.csv

    Computing power (no baseline)...

    Computing power (with baseline)...

    Reading: Density_of_segregating_sites.cs

Boundary-segment finder (for plotting boundaries of power > 0.8 on heatmaps)

In [ ]:
# Detects the boundary of the region where power >= threshold.
# Operates on ordinal grid positions (k_num, gen_num) so segment
# coordinates line up exactly with geom_tile positions.

find_boundary_segments <- function(data, threshold = 0.8) {
  k_levels   <- sort(unique(data$k))
  gen_levels <- sort(unique(data$gen_after))

  high <- data %>%
    filter(power >= threshold) %>%
    mutate(
      k_num   = match(k,         k_levels),
      gen_num = match(gen_after, gen_levels)
    )

  if (nrow(high) == 0) return(data.frame())

  hp_set <- paste(high$k_num, high$gen_num, sep = "_")

  segs <- data.frame()
  for (i in seq_len(nrow(high))) {
    kv <- high$k_num[i]
    gv <- high$gen_num[i]

    # Left edge
    if (!paste(kv - 1, gv, sep = "_") %in% hp_set)
      segs <- rbind(segs, data.frame(x = kv - 0.5, xend = kv - 0.5,
                                     y = gv - 0.5, yend = gv + 0.5))
    # Right edge
    if (!paste(kv + 1, gv, sep = "_") %in% hp_set)
      segs <- rbind(segs, data.frame(x = kv + 0.5, xend = kv + 0.5,
                                     y = gv - 0.5, yend = gv + 0.5))
    # Bottom edge
    if (!paste(kv, gv - 1, sep = "_") %in% hp_set)
      segs <- rbind(segs, data.frame(x = kv - 0.5, xend = kv + 0.5,
                                     y = gv - 0.5, yend = gv - 0.5))
    # Top edge
    if (!paste(kv, gv + 1, sep = "_") %in% hp_set)
      segs <- rbind(segs, data.frame(x = kv - 0.5, xend = kv + 0.5,
                                     y = gv + 0.5, yend = gv + 0.5))
  }
  segs
}

Build heatmaps

In [ ]:
# -- Build one heatmap tile ----------------------------------------------------

make_heatmap <- function(grid, yaxis_expr, col_label_expr,
                         show_y_axis = TRUE, show_x_axis = TRUE) {

  k_levels   <- sort(unique(grid$k))
  gen_levels <- sort(unique(grid$gen_after))

  grid <- grid %>%
    mutate(
      k_num   = match(k,         k_levels),
      gen_num = match(gen_after, gen_levels)
    )

  boundary_segs <- find_boundary_segments(grid, threshold = 0.8)

  p <- ggplot(grid, aes(x = k_num, y = gen_num, fill = power)) +
    geom_tile(color = "white", linewidth = 0.2) +
    scale_fill_viridis_c(
      option = "magma", na.value = "grey80",
      limits = c(0, 1), breaks = seq(0, 1, by = 0.2), name = "Power"
    ) +
    scale_x_continuous(breaks = seq_along(k_levels), labels = k_levels, expand = c(0,0)) +
    scale_y_continuous(breaks = seq_along(gen_levels), labels = gen_levels, expand = c(0,0)) +
    theme_minimal() +
    theme(
      panel.grid      = element_blank(),
      plot.title      = element_text(size = BASE_TITLE, face = "bold", hjust = 0.5),
      axis.title.x    = element_text(size = BASE_AXIS_T),
      axis.title.y    = element_text(size = BASE_AXIS_T),
      axis.text       = element_text(size = BASE_AXIS_TX),
      legend.title    = element_text(size = BASE_LEG_T),
      legend.text     = element_text(size = BASE_LEG_TX),
      legend.key.size = unit(BASE_KEY, "cm"),
      plot.margin     = margin(6, 6, 6, 6)
    )

  if (nrow(boundary_segs) > 0) {
    p <- p + geom_segment(data = boundary_segs,
                          aes(x = x, xend = xend, y = y, yend = yend),
                          inherit.aes = FALSE, colour = "cyan", linewidth = 1.8)
  }

  if (!is.null(col_label_expr)) {
    p <- p + ggtitle(col_label_expr)
  }

  if (show_y_axis) {
    p <- p + ylab(yaxis_expr)        # <-- uses the argument directly
  } else {
    p <- p + labs(y = NULL) +
      theme(axis.text.y = element_blank(), axis.ticks.y = element_blank())
  }

  if (show_x_axis) {
    p <- p + labs(x = "Clusters per Arm (k)")
  } else {
    p <- p + labs(x = NULL) +
      theme(axis.text.x = element_blank(), axis.ticks.x = element_blank())
  }

  p
}

# Helper: extract a plain-character stat name from the expression object
# so it can be embedded in bquote() without double-evaluation issues.
stat_info_label_for_yaxis <- function(expr_obj) {
  # Deparse the first element of the expression to a character string.
  # This gives us e.g. "LD", "Tajima's D", etc. for the top line of the
  # y-axis label.  We intentionally fall back to deparse so that special
  # characters (D in italics, pi) are handled by the caller's bquote.
  paste(deparse(expr_obj[[1]]), collapse = "")
}

# -- Assemble a 4-row x 3-col combined figure ----------------------------------

build_4x3_figure <- function(all_grids, with_baseline, dataset_pretty) {

  n_rows <- length(stat_info)   # 4
  n_cols <- length(sigma_dirs)  # 3

  plot_list <- vector("list", n_rows * n_cols)

  for (row_i in seq_len(n_rows)) {
    for (col_i in seq_len(n_cols)) {
      g <- all_grids[[col_i]][[row_i]]

      col_expr <- if (row_i == 1) sigma_label_exprs[[col_i]] else NULL
      show_y   <- (col_i == 1)
      show_x   <- (row_i == n_rows)

      p <- make_heatmap(
        grid            = g,
        yaxis_expr      = stat_info[[row_i]]$yaxis_expr,
        col_label_expr  = col_expr,
        show_y_axis     = show_y,
        show_x_axis     = show_x
      )

      plot_list[[(row_i - 1) * n_cols + col_i]] <- p
    }
  }

  baseline_str <- if (with_baseline) "With baseline" else "Without baseline"
  fig_title    <- paste0(dataset_pretty, " - ", baseline_str)

  combined <- wrap_plots(plot_list, nrow = n_rows, ncol = n_cols) +
    plot_layout(guides = "collect") +
    plot_annotation(
      theme = theme(
        plot.title      = element_text(size = BASE_ANNOT, face = "bold",
                                       hjust = 0.5),
        legend.position = "right"
      )
    )

  combined
}

In [ ]:
# -- Main loop over datasets ---------------------------------------------------

for (root in dataset_roots) {
  folder_name  <- basename(root)
  pretty_label <- dataset_label_map[[folder_name]]

  message("\n========================================")
  message("Processing: ", pretty_label, "  (", folder_name, ")")
  message("========================================")

  all_grids_no  <- vector("list", length(sigma_dirs))
  all_grids_yes <- vector("list", length(sigma_dirs))

  for (si in seq_along(sigma_dirs)) {
    sigma_path <- file.path(root, sigma_dirs[si])
    message("  Sigma dir: ", sigma_dirs[si])

    all_grids_no[[si]]  <- vector("list", length(stat_info))
    all_grids_yes[[si]] <- vector("list", length(stat_info))

    for (st in seq_along(stat_info)) {
      csv_path <- file.path(sigma_path, stat_info[[st]]$file)
      message("    Reading: ", stat_info[[st]]$file)
      df <- read.csv(csv_path)

      message("    Computing power (no baseline)...")
      all_grids_no[[si]][[st]]  <- compute_power_grid(df, with_baseline = FALSE)

      message("    Computing power (with baseline)...")
      all_grids_yes[[si]][[st]] <- compute_power_grid(df, with_baseline = TRUE)
    }
  }

  # Save figures
  out_dir <- file.path("../outputs", folder_name)
  dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

  message("  Building without-baseline figure...")
  fig_no <- build_4x3_figure(all_grids_no,  with_baseline = FALSE,
                              dataset_pretty = pretty_label)
  out_no <- file.path(out_dir, "nobaseline.pdf")
  ggsave(out_no, fig_no, width = 20, height = 26, units = "in", dpi = 300)
  message("  Saved: ", out_no)

  message("  Building with-baseline figure...")
  fig_yes <- build_4x3_figure(all_grids_yes, with_baseline = TRUE,
                               dataset_pretty = pretty_label)
  out_yes <- file.path(out_dir, "baseline.pdf")
  ggsave(out_yes, fig_yes, width = 20, height = 26, units = "in", dpi = 300)
  message("  Saved: ", out_yes)

  message("  Done: ", pretty_label)
}

message("\nAll datasets processed.")